# 12 build_combined_scores

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [ ]:
LAMBDA_MIN = pd.read_csv("HNSC_hypoxia_signature_lambda_min.csv")
LAMBDA_1SE = pd.read_csv("HNSC_hypoxia_signature_lambda_1se.csv")

TOUSTRUP_GENES = ["ADM","ALDOA","ANKRD37","BNIP3","BNIP3L","FAM162A","EGLN3","KCTD11",
                   "LOX","NDRG1","P4HA1","P4HA2","PDK1","PFKFB3","SLC2A1"]
EUSTACE_GENES = ["ALDOA","ANGPTL4","ANLN","BNC1","MRGBP","CA9","CDKN3","COL4A6","DCBLD1","ENO1",
                  "FAM83B","FOSL1","GNAI1","HILPDA","KCTD11","KRT17","LDHA","MRPS17","P4HA1",
                  "PGAM1","PGK1","SDC1","SLC16A1","SLC2A1","TPI1","VEGFA"]  # full 26-gene panel
EUSTACE_CONTROL_GENES = ["GNB2L1", "B2M", "RPL11"]
LENDAHL_GENES = ["ADM","ALDOC","BNIP3","BNIP3L","DDIT4","ENO2","ERO1L","GBE1","GLRX",
                  "HK2","HMMR","INSIG1","INSIG2","KDM3A","MAFF","MXI1","MYCBP","NDRG1",
                  "P4HA1","PDK1","PPFIA4","SLC26A2","SLC7A1","SOX4","ATF7IP","VEGFA","WSB1","ZNF654"]  # 28 of 30: HIG2 excluded (genuinely absent), DHX34 excluded (unresolved "(BRCA1)" annotation, no legitimate HGNC alias found)
RSI_GENES = ["AR", "JUN", "STAT1", "PRKCB", "RELA", "ABL1", "SUMO1", "PAK2", "HDAC1", "IRF1"]
RSI_COEFFICIENTS = {
    "AR": -0.0098009, "JUN": 0.0128283, "STAT1": 0.0254552, "PRKCB": -0.0017589,
    "RELA": -0.0038171, "ABL1": 0.1070213, "SUMO1": -0.0002509, "PAK2": -0.0092431,
    "HDAC1": -0.0204469, "IRF1": -0.0441683,
}

# Platform-era alias resolution (GPL10558 predates some HGNC renames)
PLATFORM_ALIAS = {"MRGBP": "C20orf20", "KDM3A": "JMJD1A"}

all_genes_needed = set(LAMBDA_MIN["Gene"]) | set(LAMBDA_1SE["Gene"]) | set(TOUSTRUP_GENES) | \
                   set(EUSTACE_GENES) | set(LENDAHL_GENES) | set(RSI_GENES) | set(EUSTACE_CONTROL_GENES)
print("Total unique genes needed across all 6 signatures:", len(all_genes_needed))

In [ ]:
probe_to_symbol = {}
with open("GPL10558_HumanHT-12_V4_0_R2_15002873_B.txt", "r", encoding="latin-1") as f:
    for line in f:
        if line.startswith("Species\tSource\tSearch_Key"):
            header = line.strip().split("\t")
            break
    probe_idx = header.index("Probe_Id")
    symbol_idx = header.index("Symbol")
    for line in f:
        if line.startswith("[Controls]") or line.strip() == "":
            break
        fields = line.rstrip("\n").split("\t")
        if len(fields) <= max(probe_idx, symbol_idx):
            continue
        pid, sym = fields[probe_idx].strip(), fields[symbol_idx].strip()
        if pid and sym:
            probe_to_symbol[pid] = sym

needed_platform_symbols = {PLATFORM_ALIAS.get(g, g) for g in all_genes_needed}
symbol_to_probe_ids = defaultdict(list)
for pid, sym in probe_to_symbol.items():
    if sym in needed_platform_symbols:
        symbol_to_probe_ids[sym].append(pid)

needed_probe_ids = set(p for probes in symbol_to_probe_ids.values() for p in probes)
print("Probes to extract:", len(needed_probe_ids))

In [ ]:
wb = openpyxl.load_workbook("GSE65858.xlsx", data_only=True, read_only=True)
ws = wb["Sheet1"]
header_row = list(ws.iter_rows(min_row=105, max_row=105, values_only=True))[0]
sample_ids = [str(v) for v in header_row[1:] if v]

expression_rows = {}
for row in ws.iter_rows(min_row=106, max_row=ws.max_row, values_only=True):
    if row[0] in needed_probe_ids:
        expression_rows[row[0]] = row[1:1+len(sample_ids)]

gene_expr = {}
for g in all_genes_needed:
    platform_symbol = PLATFORM_ALIAS.get(g, g)
    probes = symbol_to_probe_ids.get(platform_symbol, [])
    vals = [expression_rows[p] for p in probes if p in expression_rows]
    if vals:
        gene_expr[g] = np.array(vals, dtype=float).mean(axis=0)  # fixed rule: mean across multi-probes

expr_df = pd.DataFrame(gene_expr, index=sample_ids)
print("Gene-level expression matrix:", expr_df.shape)

In [ ]:
def get_row_values(row_idx, n_samples):
    row = list(ws.iter_rows(min_row=row_idx, max_row=row_idx, max_col=1+n_samples, values_only=True))[0]
    vals = []
    for v in row[1:]:
        if v is None:
            vals.append(None); continue
        v = str(v)
        vals.append(v.split(":", 1)[1].strip() if ":" in v else v)
    return vals

n = len(sample_ids)
os_time = get_row_values(51, n)
os_event = get_row_values(52, n)
clin = pd.DataFrame({"Sample": sample_ids, "OS_time_days": os_time, "OS_event_raw": os_event}).set_index("Sample")
clin["OS_time_days"] = pd.to_numeric(clin["OS_time_days"], errors="coerce")
clin["OS_months"] = clin["OS_time_days"] / 30.44
clin["Event"] = clin["OS_event_raw"].map({"TRUE": 1, "FALSE": 0})

data = expr_df.join(clin[["OS_months", "Event"]], how="inner")
data = data.dropna(subset=["OS_months", "Event"])
data = data[data["OS_months"] > 0]
print("Patients with expression + survival:", data.shape[0])

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std()

# ---- 1. Our lambda.min (locked coefficients, no refitting) ----
score = pd.Series(0.0, index=data.index)
for _, row in LAMBDA_MIN.iterrows():
    score += row["Coefficient"] * zscore(data[row["Gene"]])
data["Score_lambda_min"] = score - score.mean()

# ---- 2. Our lambda.1se (locked coefficients, no refitting) ----
score = pd.Series(0.0, index=data.index)
for _, row in LAMBDA_1SE.iterrows():
    score += row["Coefficient"] * zscore(data[row["Gene"]])
data["Score_lambda_1se"] = score - score.mean()

# ---- 3. Toustrup (equal-weighted mean -- no published weights) ----
z = data[TOUSTRUP_GENES].apply(zscore)
data["Score_Toustrup"] = z.mean(axis=1)
data["Score_Toustrup"] -= data["Score_Toustrup"].mean()

# ---- 4. Eustace/NIMRAD (median-of-panel minus mean-of-controls, per the
# original paper's method, adapted from qPCR Ct to log2 expression) ----
eustace_genes_avail = [g for g in EUSTACE_GENES if g in data.columns]
panel_z = data[eustace_genes_avail].apply(zscore)
control_z = data[EUSTACE_CONTROL_GENES].apply(zscore)
data["Score_Eustace"] = panel_z.median(axis=1) - control_z.mean(axis=1)
data["Score_Eustace"] -= data["Score_Eustace"].mean()

# ---- 5. Lendahl (equal-weighted mean -- no published weights, per Tawk et al.) ----
lendahl_genes_avail = [g for g in LENDAHL_GENES if g in data.columns]
z = data[lendahl_genes_avail].apply(zscore)
data["Score_Lendahl"] = z.mean(axis=1)
data["Score_Lendahl"] -= data["Score_Lendahl"].mean()

# ---- 6. RSI -- CORRECTED: rank the 10 genes WITHIN each patient (row-wise,
# 1-10), not across patients. This is the fix from Jimmy's correction round. ----
ranks = data[RSI_GENES].rank(axis=1, method="average")
assert (ranks.sum(axis=1).round(6) == 55.0).all(), "Sanity check failed: each patient's 10 ranks should sum to 55"
data["Score_RSI"] = sum(RSI_COEFFICIENTS[g] * ranks[g] for g in RSI_GENES)

print("All 6 scores computed.")
print("Sanity check passed: every patient's RSI ranks sum to 55 (1+2+...+10), confirming within-patient ranking.")

In [ ]:
output_cols = ["OS_months", "Event", "Score_lambda_min", "Score_lambda_1se",
               "Score_Toustrup", "Score_Eustace", "Score_Lendahl", "Score_RSI"]
patient_scores = data[output_cols].copy()
patient_scores.index.name = "GSM_Sample_ID"
patient_scores.to_csv("GSE65858_all_signatures_patient_scores.csv")
print("Saved GSE65858_all_signatures_patient_scores.csv:", patient_scores.shape)
patient_scores.head()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from lifelines import CoxPHFitter

expected = {
    "Score_lambda_min": "HR=1.81, p=0.016",
    "Score_lambda_1se": "HR=2.65, p=0.024",
    "Score_Toustrup": "HR=1.27, p=0.179",
    "Score_Eustace": "HR=0.99, p=0.94",
    "Score_Lendahl": "HR=1.42, p=0.172 (28 genes, DHX34 excluded)",
    "Score_RSI": "HR=3.18, p=0.245 (corrected ranking)",
}

for col, exp in expected.items():
    d = patient_scores[["OS_months", "Event", col]].dropna()
    cph = CoxPHFitter()
    cph.fit(d, duration_col="OS_months", event_col="Event")
    s = cph.summary.loc[col]
    hr = np.exp(s["coef"])
    print(f"{col}: HR={hr:.3f}, p={s['p']:.4g}  (expected: {exp})")